<a href="https://colab.research.google.com/github/TAU-CH/midrash_hebrew_script_mode_classifier/blob/main/Hebrew_Script_Mode_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ▶ HEBREW SCRIPT MODE CLASSIFIER { display-mode: "form" }
# Click the play button once. The interface will appear below.

import logging
import os
import subprocess
import sys
import warnings
from pathlib import Path

from IPython.display import clear_output


# ------------------------------------------------------------
# 1. Install dependencies
# ------------------------------------------------------------

try:
    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "timm>=1.0,<2",
            "gradio>=6.0,<7",
            "huggingface_hub>=0.30,<2",
        ],
        check=True,
        capture_output=True,
        text=True,
    )

except subprocess.CalledProcessError as error:
    error_message = (
        error.stderr[-2000:]
        if error.stderr
        else "No installation details were returned."
    )

    raise RuntimeError(
        "The required packages could not be installed. "
        "Reconnect the Colab runtime and try again.\n\n"
        f"{error_message}"
    ) from error


# ------------------------------------------------------------
# 2. Suppress nonessential warnings and progress output
# ------------------------------------------------------------

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module=r"timm(\..*)?",
)

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module=r"gradio(\..*)?",
)

warnings.filterwarnings(
    "ignore",
    message=r".*unauthenticated requests to the HF Hub.*",
)

for logger_name in [
    "huggingface_hub",
    "huggingface_hub.utils._http",
]:
    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.ERROR)
    logger.propagate = False


# Import Hugging Face only after configuring its environment.
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import logging as hf_logging

hf_logging.set_verbosity_error()


# ------------------------------------------------------------
# 3. Download the public model checkpoint
# ------------------------------------------------------------

print(
    "Preparing the classifier. "
    "The first run may take a few minutes..."
)

try:
    CHECKPOINT_PATH = Path(
        hf_hub_download(
            repo_id="beratkurar/hebrew_script_mode_classifier",
            filename="hsmc_model.pt",
        )
    )

except Exception as error:
    raise RuntimeError(
        "The trained model could not be downloaded from Hugging Face. "
        "Check the internet connection and try again."
    ) from error

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f"The downloaded checkpoint was not found: {CHECKPOINT_PATH}"
    )


# ------------------------------------------------------------
# 4. Import inference dependencies
# ------------------------------------------------------------

import math

import numpy as np
from PIL import Image
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import functional as TF


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


class MaskedGatedAttentionPool(nn.Module):
    def __init__(self, in_dim, hidden_dim=512, dropout=0.1):
        super().__init__()
        self.att_v = nn.Linear(in_dim, hidden_dim)
        self.att_u = nn.Linear(in_dim, hidden_dim)
        self.att_w = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, feat, mask):
        batch_size, channels, height, width = feat.shape
        tokens = feat.flatten(2).transpose(1, 2)
        valid_tokens = mask.flatten(1)

        scores = torch.tanh(self.att_v(tokens)) * torch.sigmoid(
            self.att_u(tokens)
        )
        scores = self.att_w(self.dropout(scores)).squeeze(-1)
        scores = scores.masked_fill(
            ~valid_tokens,
            torch.finfo(scores.dtype).min,
        )

        attention = torch.softmax(scores, dim=1)
        pooled = torch.sum(tokens * attention.unsqueeze(-1), dim=1)

        return pooled, attention.view(batch_size, height, width)


class HSCDConvNeXtAttention(nn.Module):
    def __init__(
        self,
        backbone_name,
        num_classes=2,
        feature_out_index=3,
        dropout=0.3,
    ):
        super().__init__()

        self.backbone = timm.create_model(
            backbone_name,
            pretrained=False,
            features_only=True,
            out_indices=(int(feature_out_index),),
        )

        feature_dim = self.backbone.feature_info.channels()[0]

        self.pool = MaskedGatedAttentionPool(
            feature_dim,
            hidden_dim=512,
            dropout=dropout,
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Dropout(dropout),
            nn.Linear(feature_dim, num_classes),
        )

    def forward(self, images, pixel_mask):
        features = self.backbone(images)[-1]

        feature_mask = F.interpolate(
            pixel_mask.float().unsqueeze(1),
            size=features.shape[-2:],
            mode="nearest",
        ).squeeze(1).bool()

        pooled, attention = self.pool(features, feature_mask)
        logits = self.classifier(pooled)

        return logits, attention


if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

config = checkpoint["config"]
CLASS_NAMES = checkpoint.get(
    "classes",
    ["non_square", "square"],
)

MAX_HEIGHT = int(config.get("IMG_CROP_MAX_HEIGHT", 2500))
MAX_WIDTH = int(config.get("IMG_CROP_MAX_WIDTH", 2500))

model = HSCDConvNeXtAttention(
    backbone_name=config.get(
        "BACKBONE_NAME",
        "convnext_base.fb_in22k_ft_in1k",
    ),
    num_classes=len(CLASS_NAMES),
    feature_out_index=config.get("BACKBONE_OUT_INDEX", 3),
    dropout=config.get("DROPOUT", 0.3),
)

model.load_state_dict(checkpoint["model_state"])
model.to(DEVICE)
model.eval()

clear_output(wait=True)

IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406]
).view(3, 1, 1)

IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225]
).view(3, 1, 1)


def center_crop_to_max_size(
    image,
    max_height=MAX_HEIGHT,
    max_width=MAX_WIDTH,
):
    width, height = image.size

    crop_height = min(height, max_height)
    crop_width = min(width, max_width)

    left = int(round((width - crop_width) / 2))
    top = int(round((height - crop_height) / 2))

    return image.crop(
        (
            left,
            top,
            left + crop_width,
            top + crop_height,
        )
    )


def load_pil_image(image):
    """
    Accept a PIL image or a filesystem path and return an RGB PIL image.
    """
    if isinstance(image, Image.Image):
        return image.convert("RGB")

    image_path = Path(image).expanduser()

    if not image_path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")

    return Image.open(image_path).convert("RGB")


def prepare_image(image):
    """
    Apply exactly the preprocessing used by the original inference notebook.
    """
    image = load_pil_image(image)
    image = center_crop_to_max_size(image)

    tensor = TF.to_tensor(image)
    tensor = (tensor - IMAGENET_MEAN) / IMAGENET_STD

    _, height, width = tensor.shape

    padded_height = int(math.ceil(height / 32) * 32)
    padded_width = int(math.ceil(width / 32) * 32)

    normalized_white = (
        torch.ones(3, 1, 1) - IMAGENET_MEAN
    ) / IMAGENET_STD

    canvas = normalized_white.expand(
        3,
        padded_height,
        padded_width,
    ).clone()

    canvas[:, :height, :width] = tensor

    mask = torch.zeros(
        padded_height,
        padded_width,
        dtype=torch.bool,
    )
    mask[:height, :width] = True

    return canvas.unsqueeze(0), mask.unsqueeze(0)


@torch.inference_mode()
def predict_image(image):
    image_tensor, pixel_mask = prepare_image(image)

    image_tensor = image_tensor.to(DEVICE)
    pixel_mask = pixel_mask.to(DEVICE)

    with torch.amp.autocast(
        "cuda",
        enabled=DEVICE.type == "cuda",
    ):
        logits, _ = model(image_tensor, pixel_mask)

    probabilities = torch.softmax(
        logits.float(),
        dim=1,
    )[0].cpu().numpy()

    predicted_index = int(probabilities.argmax())

    probability_dict = {
        class_name: float(probabilities[index])
        for index, class_name in enumerate(CLASS_NAMES)
    }

    return CLASS_NAMES[predicted_index], probability_dict


import math

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
import torch
import torch.nn.functional as F
from torchvision.transforms import functional as TF


def prepare_attention_input(image):
    """
    Accept a PIL image or image path and reproduce the model preprocessing.
    Returns:
        display_image: cropped RGB PIL image
        input_tensor:  [1, 3, padded_height, padded_width]
        pixel_mask:    [1, padded_height, padded_width]
    """
    if isinstance(image, Image.Image):
        display_image = image.convert("RGB")
    else:
        display_image = Image.open(image).convert("RGB")

    display_image = center_crop_to_max_size(display_image)

    tensor = TF.to_tensor(display_image)
    tensor = (tensor - IMAGENET_MEAN) / IMAGENET_STD

    _, height, width = tensor.shape

    padded_height = int(math.ceil(height / 32) * 32)
    padded_width = int(math.ceil(width / 32) * 32)

    normalized_white = (
        torch.ones(3, 1, 1) - IMAGENET_MEAN
    ) / IMAGENET_STD

    canvas = normalized_white.expand(
        3,
        padded_height,
        padded_width,
    ).clone()

    canvas[:, :height, :width] = tensor

    pixel_mask = torch.zeros(
        padded_height,
        padded_width,
        dtype=torch.bool,
    )
    pixel_mask[:height, :width] = True

    return (
        display_image,
        canvas.unsqueeze(0),
        pixel_mask.unsqueeze(0),
    )


@torch.inference_mode()
def create_attention_overlay(
    image,
    overlay_alpha=0.70,
    cmap="turbo",
    upper_percentile=99.5,
):
    """
    Run inference and return:
        overlay_image: PIL image containing the attention overlay
        prediction: predicted class name
        probabilities: dictionary of class probabilities
        attention_map: normalized 2D NumPy attention map
    """
    display_image, input_tensor, pixel_mask = (
        prepare_attention_input(image)
    )

    image_width, image_height = display_image.size

    padded_height = input_tensor.shape[-2]
    padded_width = input_tensor.shape[-1]

    input_tensor = input_tensor.to(DEVICE)
    pixel_mask = pixel_mask.to(DEVICE)

    with torch.amp.autocast(
        "cuda",
        enabled=DEVICE.type == "cuda",
    ):
        logits, attention = model(
            input_tensor,
            pixel_mask,
        )

    probabilities_array = torch.softmax(
        logits.float(),
        dim=1,
    )[0].cpu().numpy()

    predicted_index = int(probabilities_array.argmax())
    prediction = CLASS_NAMES[predicted_index]

    probabilities = {
        class_name: float(probabilities_array[index])
        for index, class_name in enumerate(CLASS_NAMES)
    }

    # attention initially has shape:
    # [batch, feature_height, feature_width]
    #
    # Upsample it to the padded input-image resolution.
    attention_upscaled = F.interpolate(
        attention.float().unsqueeze(1),
        size=(padded_height, padded_width),
        mode="bilinear",
        align_corners=False,
    )[0, 0]

    # Remove the padding area.
    attention_map = attention_upscaled[
        :image_height,
        :image_width,
    ].cpu().numpy()

    # Robust normalization prevents one extreme value from
    # suppressing the rest of the visualization.
    lower_value = float(attention_map.min())
    upper_value = float(
        np.percentile(
            attention_map,
            upper_percentile,
        )
    )

    if upper_value > lower_value:
        attention_map = (
            attention_map - lower_value
        ) / (upper_value - lower_value)
    else:
        attention_map = np.zeros_like(
            attention_map,
            dtype=np.float32,
        )

    attention_map = np.clip(
        attention_map,
        0.0,
        1.0,
    )

    original_array = (
        np.asarray(display_image).astype(np.float32) / 255.0
    )

    colormap = plt.get_cmap(cmap)

    heatmap_rgb = colormap(
        attention_map
    )[..., :3].astype(np.float32)

    # Strong-attention regions receive more overlay;
    # low-attention regions remain close to the original image.
    alpha_map = (
        overlay_alpha
        * attention_map[..., np.newaxis]
    )

    overlay_array = (
        original_array * (1.0 - alpha_map)
        + heatmap_rgb * alpha_map
    )

    overlay_array = np.clip(
        overlay_array * 255.0,
        0,
        255,
    ).astype(np.uint8)

    overlay_image = Image.fromarray(overlay_array)

    return (
        overlay_image,
        prediction,
        probabilities,
        attention_map,
    )


def show_attention(
    image,
    overlay_alpha=0.70,
    cmap="turbo",
):
    overlay_image, prediction, probabilities, _ = (
        create_attention_overlay(
            image=image,
            overlay_alpha=overlay_alpha,
            cmap=cmap,
        )
    )

    confidence = probabilities[prediction]

    plt.figure(figsize=(12, 12))
    plt.imshow(overlay_image)
    plt.title(
        f"Prediction: {prediction} | "
        f"Confidence: {confidence:.2%}"
    )
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    print("Prediction:", prediction)
    print("Probabilities:")

    for class_name, probability in probabilities.items():
        print(f"  {class_name}: {probability:.4%}")

    return overlay_image

from IPython.display import clear_output
import gradio as gr

# Remove installation, download, and model-loading logs before showing the UI.
clear_output(wait=True)

DISPLAY_NAMES = {
    "square": "Square",
    "non_square": "Non-square",
}


def classify_with_attention(image):
    if image is None:
        raise gr.Error("Please upload a document image first.")

    try:
        (
            overlay_image,
            prediction,
            probabilities,
            _,
        ) = create_attention_overlay(image)

    except RuntimeError as error:
        if "out of memory" in str(error).lower():
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            raise gr.Error(
                "This image requires too much memory. "
                "Please upload a smaller image."
            ) from error

        raise

    label_probabilities = {
        DISPLAY_NAMES.get(class_name, class_name): probability
        for class_name, probability in probabilities.items()
    }

    return label_probabilities, overlay_image


CUSTOM_CSS = """
.gradio-container {
    max-width: 1180px !important;
    margin: 0 auto !important;
}
.main-title {
    text-align: center;
}
.instructions {
    text-align: center;
    margin-bottom: 1rem;
}
"""

with gr.Blocks(
    title="Hebrew Script Mode Classifier",
) as demo:
    gr.Markdown(
        """
        <div class="main-title">

        # Hebrew Script Mode Classifier

        </div>

        <div class="instructions">

        Upload a Hebrew manuscript image. Analysis starts automatically.
        The classifier predicts whether the script mode is
        **Square** or **Non-square** and displays its attention heatmap.

        </div>
        """
    )

    with gr.Row():
        input_image = gr.Image(
            type="pil",
            image_mode="RGB",
            sources=["upload"],
            label="Upload a manuscript image",
            height=520,
        )

        attention_output = gr.Image(
            type="pil",
            label="Attention heatmap",
            height=520,
            interactive=False,
        )

    prediction_output = gr.Label(
        num_top_classes=len(CLASS_NAMES),
        label="Script mode prediction",
    )

    clear_button = gr.Button(
        "Clear",
        size="lg",
    )

    # Start inference immediately after the user uploads an image.
    input_image.upload(
        fn=classify_with_attention,
        inputs=input_image,
        outputs=[prediction_output, attention_output],
        show_progress="full",
    )

    # Also clear the outputs if the user removes the image with the
    # built-in clear control inside the image component.
    input_image.clear(
        fn=lambda: (None, None),
        inputs=None,
        outputs=[
            prediction_output,
            attention_output,
        ],
    )

    clear_button.click(
        fn=lambda: (None, None, None),
        inputs=None,
        outputs=[
            input_image,
            prediction_output,
            attention_output,
        ],
    )

# Launch Gradio, print the public URL, and try to open it only now,
# after the user has explicitly run this cell.
import contextlib
import io
import json as _json

_launch_log = io.StringIO()

with contextlib.redirect_stdout(_launch_log):
    _server_app, _local_url, _share_url = demo.queue(
        default_concurrency_limit=1,
    ).launch(
        share=True,
        inline=False,
        debug=False,
        show_error=True,
        quiet=True,
        theme=gr.themes.Soft(),
        css=CUSTOM_CSS,
        footer_links=[],
    )

if not _share_url:
    raise RuntimeError(
        "Gradio could not create a public link. "
        "Reconnect the Colab runtime and run the start cell again."
    )

print(f"* Running on public URL: {_share_url}")

# Display a prominent runtime-only button. Because notebook outputs are
# omitted when saving, this button is created only after this cell runs
# and always points to the current live Gradio session.
import html as _html
from IPython.display import HTML, display

_safe_share_url = _html.escape(_share_url, quote=True)

display(
    HTML(
        f"""
        <div style="
            max-width: 680px;
            margin: 22px auto;
            padding: 24px;
            text-align: center;
            border: 1px solid #d9dce1;
            border-radius: 16px;
            background: #ffffff;
            box-shadow: 0 4px 14px rgba(0, 0, 0, 0.08);
            font-family: Arial, sans-serif;
        ">

            <a
                href="{_safe_share_url}"
                target="_blank"
                rel="noopener noreferrer"
                style="
                    display: inline-block;
                    min-width: 250px;
                    padding: 15px 28px;
                    border-radius: 10px;
                    background: #2563eb;
                    color: #ffffff;
                    text-decoration: none;
                    font-size: 18px;
                    font-weight: 700;
                    line-height: 1.2;
                "
            >
                Open classifier
            </a>
        </div>
        """
    )
)

# Try to open the same fresh URL automatically after the server is ready.
try:
    from google.colab.output import eval_js

    _opened = eval_js(
        f"Boolean(window.open({_json.dumps(_share_url)}, '_blank'))"
    )

    if not _opened:
        print("Your browser blocked the new tab. Use the blue button above.")
except Exception:
    print("Use the blue button above to open the classifier.")
